# Batched Report Pipeline Validation

This notebook validates the centralized batched solving flow end to end against Unity Catalog-backed integration data.

It checks these pieces together against a temporary subset copied from Unity Catalog source tables:
- `create_sink`
- `_cleanup_temp_tables`
- `get_selectors()`
- `batch_size` propagation
- `build_batches`
- `_solve_expressions_batched`
- `determine_report`
- `determine_events`
- `determine_aggregations`

Update the catalog, schema, or table prefix settings in Cell 2 before running if your Databricks workspace uses different names.

In [0]:
from pathlib import Path
import sys

def resolve_module_path() -> Path:
    cwd = Path.cwd().resolve()
    for root in (cwd, *cwd.parents):
        repo_candidate = root / "shared_components" / "src" / "mda_framework" / "src"
        if repo_candidate.exists():
            return repo_candidate

        framework_candidate = root / "src"
        if root.name == "mda_framework" and framework_candidate.exists():
            return framework_candidate

    raise FileNotFoundError(
        "Could not resolve shared_components/src/mda_framework/src from the current working directory"
    )

MDA_FRAMEWORK_SRC = str(resolve_module_path())
if MDA_FRAMEWORK_SRC not in sys.path:
    sys.path.insert(0, MDA_FRAMEWORK_SRC)

dbutils_ref = globals().get("dbutils")
if dbutils_ref is not None:
    dbutils_ref.widgets.removeAll()
    dbutils_ref.widgets.text("source_catalog", "development", "Source catalog")
    dbutils_ref.widgets.text("source_schema", "silver", "Source schema")
    dbutils_ref.widgets.text("sink_schema", "gold_e2e", "Sink schema")
    dbutils_ref.widgets.text("table_prefix", "batched_e2e", "Table prefix")
    SOURCE_CATALOG = dbutils_ref.widgets.get("source_catalog").strip()
    SOURCE_SCHEMA = dbutils_ref.widgets.get("source_schema").strip()
    SINK_SCHEMA = dbutils_ref.widgets.get("sink_schema").strip()
    TABLE_PREFIX = dbutils_ref.widgets.get("table_prefix").strip()
else:
    SOURCE_CATALOG = "development"
    SOURCE_SCHEMA = "silver"
    SINK_SCHEMA = "gold_e2e"
    TABLE_PREFIX = "batched_e2e"

ENGINE_RPM_CHANNEL_NAME = "is1_eng_speed"
VEHICLE_SPEED_CHANNEL_NAME = "can_vehicle_speed"
CHANNEL_DATA_KEY = "TM"

SILVER_NAMESPACE = f"{SOURCE_CATALOG}.{SOURCE_SCHEMA}"
GOLD_NAMESPACE = f"{SOURCE_CATALOG}.{SINK_SCHEMA}"

SOURCE_CONTAINER_METRICS_TABLE = f"{SILVER_NAMESPACE}.container_metric"
SOURCE_CHANNEL_METRICS_TABLE = f"{SILVER_NAMESPACE}.channel_metric"
SOURCE_CHANNELS_TABLE = f"{SILVER_NAMESPACE}.channel_data"

TEMP_CONTAINER_METRICS_TABLE = f"{GOLD_NAMESPACE}.{TABLE_PREFIX}_container_metric_src"
TEMP_CHANNEL_METRICS_TABLE = f"{GOLD_NAMESPACE}.{TABLE_PREFIX}_channel_metric_src"
TEMP_CHANNELS_TABLE = f"{GOLD_NAMESPACE}.{TABLE_PREFIX}_channel_data_src"

print(
    {
        "MDA_FRAMEWORK_SRC": MDA_FRAMEWORK_SRC,
        "SOURCE_CONTAINER_METRICS_TABLE": SOURCE_CONTAINER_METRICS_TABLE,
        "SOURCE_CHANNEL_METRICS_TABLE": SOURCE_CHANNEL_METRICS_TABLE,
        "SOURCE_CHANNELS_TABLE": SOURCE_CHANNELS_TABLE,
        "GOLD_NAMESPACE": GOLD_NAMESPACE,
        "TABLE_PREFIX": TABLE_PREFIX,
        "ENGINE_RPM_CHANNEL_NAME": ENGINE_RPM_CHANNEL_NAME,
        "VEHICLE_SPEED_CHANNEL_NAME": VEHICLE_SPEED_CHANNEL_NAME,
        "CHANNEL_DATA_KEY": CHANNEL_DATA_KEY,
    }
)

In [0]:
spark = globals().get("spark")
if spark is not None:
    print("Using existing Spark session")
else:
    from delta import configure_spark_with_delta_pip
    from pyspark.sql import SparkSession

    spark = configure_spark_with_delta_pip(
        SparkSession.builder.master("local[*]")
        .config(
            "spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog",
        )
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.shuffle.partitions", 1)
    ).getOrCreate()
    print("Created local Spark session")

In [0]:
from collections import Counter
import types

import pyspark.sql.functions as F

import mda_reporting.core.report as report_module
from mda_reporting.aggregations.histogram import Histogram, HistogramDuration
from mda_reporting.aggregations.stats_aggregator import StatsAggregator
from mda_reporting.config.config_parser import (
    MdaConfig,
    MeasurementDimensions,
    QueryEngine,
    Solvers,
    Source,
    UnitySink,
)
from mda_reporting.core.page import Page
from mda_reporting.core.report import Report
from mda_reporting.events.basic_event import BasicEvent

In [0]:
def setup_uc_tables() -> dict[str, object]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_NAMESPACE}")

    for row in spark.sql(f"SHOW TABLES IN {GOLD_NAMESPACE} LIKE '{TABLE_PREFIX}_*'").collect():
        spark.sql(f"DROP TABLE IF EXISTS {GOLD_NAMESPACE}.`{row.tableName}`")

    for row in spark.sql(f"SHOW TABLES IN {GOLD_NAMESPACE} LIKE '__mda_temp_*'").collect():
        spark.sql(f"DROP TABLE IF EXISTS {GOLD_NAMESPACE}.`{row.tableName}`")

    for table_name in [
        SOURCE_CONTAINER_METRICS_TABLE,
        SOURCE_CHANNEL_METRICS_TABLE,
        SOURCE_CHANNELS_TABLE,
    ]:
        try:
            spark.read.table(table_name).limit(1).collect()
        except Exception as exc:
            raise AssertionError(f"Missing Unity Catalog table: {table_name}") from exc

    container_ids = [
        row.container_id
        for row in spark.read.table(SOURCE_CONTAINER_METRICS_TABLE)
        .select("container_id")
        .distinct()
        .orderBy("container_id")
        .limit(2)
        .collect()
    ]
    assert container_ids, f"No container_ids found in {SOURCE_CONTAINER_METRICS_TABLE}"

    container_metrics_df = spark.read.table(SOURCE_CONTAINER_METRICS_TABLE).where(
        F.col("container_id").isin(container_ids)
    )
    channel_metrics_df = spark.read.table(SOURCE_CHANNEL_METRICS_TABLE).where(
        F.col("container_id").isin(container_ids)
    )
    channels_df = spark.read.table(SOURCE_CHANNELS_TABLE)

    container_metric_count = container_metrics_df.count()
    channel_metric_count = channel_metrics_df.count()
    channels_count = channels_df.count()

    assert container_metric_count > 0, "Expected container_metric rows in the selected UC subset"
    assert channel_metric_count > 0, "Expected channel_metric rows in the selected UC subset"
    assert channels_count > 0, f"Expected rows in {SOURCE_CHANNELS_TABLE}"

    container_metrics_df.write.format("delta").mode("overwrite").saveAsTable(
        TEMP_CONTAINER_METRICS_TABLE
    )
    channel_metrics_df.write.format("delta").mode("overwrite").saveAsTable(
        TEMP_CHANNEL_METRICS_TABLE
    )
    channels_df.write.format("delta").mode("overwrite").saveAsTable(
        TEMP_CHANNELS_TABLE
    )

    return {
        "container_ids": container_ids,
        "container_metric_rows": container_metric_count,
        "channel_metric_rows": channel_metric_count,
        "channels_rows": channels_count,
        "temp_container_metrics_table": TEMP_CONTAINER_METRICS_TABLE,
        "temp_channel_metrics_table": TEMP_CHANNEL_METRICS_TABLE,
        "temp_channels_table": TEMP_CHANNELS_TABLE,
    }


def build_batched_report() -> tuple[Report, dict[str, object]]:
    config = MdaConfig(
        source=Source(
            container_metrics_table=TEMP_CONTAINER_METRICS_TABLE,
            channel_metrics_table=TEMP_CHANNEL_METRICS_TABLE,
            channels_uri=TEMP_CHANNELS_TABLE,
        ),
        unity_sink=UnitySink(
            catalog=SOURCE_CATALOG,
            schema=SINK_SCHEMA,
            table_prefix=TABLE_PREFIX,
        ),
        query_engine=QueryEngine(
            solver=Solvers.BASIC_NARROW_SOLVER,
            batch_size=1,
        ),
        measurement_dimensions=[
            MeasurementDimensions.CONTAINER_ID,
            MeasurementDimensions.UUT_ID,
            MeasurementDimensions.START_TS,
            MeasurementDimensions.STOP_TS,
        ],
    )

    report = Report(
        name=f"{TABLE_PREFIX}_batched_pipeline_report",
        spark=spark,
        config=dict(config),
    )
    query = report.get_db().query

    engine_rpm = query.channel(
        channel_name=ENGINE_RPM_CHANNEL_NAME,
        data_key=CHANNEL_DATA_KEY,
    )
    vehicle_speed = query.channel(
        channel_name=VEHICLE_SPEED_CHANNEL_NAME,
        data_key=CHANNEL_DATA_KEY,
    )

    moving_vehicle_event = BasicEvent(
        name="moving_vehicle_event",
        expr=vehicle_speed > 1,
        desc="Vehicle speed above 1 km/h",
    )
    engine_running_event = BasicEvent(
        name="engine_running_event",
        expr=engine_rpm > 500,
        desc="Engine RPM above idle",
    )

    report.add_event(moving_vehicle_event)
    report.add_event(engine_running_event)

    page = Page(page_number=1)
    report.add_page(page)

    rpm_histogram = HistogramDuration(
        name="engine_rpm_hist_batched",
        base_expr=engine_rpm,
        bins=[float(i) for i in range(0, 8000, 250)],
    )
    page.add_aggregation(rpm_histogram)

    speed_rpm_stats = StatsAggregator(
        name="batched_speed_rpm_stats",
        input_expressions=[engine_rpm, vehicle_speed],
        channel_names=[ENGINE_RPM_CHANNEL_NAME, VEHICLE_SPEED_CHANNEL_NAME],
        statistics=["min", "max", "mean"],
    )
    page.add_aggregation(speed_rpm_stats)

    tracked_expressions = {
        "moving_vehicle_event": moving_vehicle_event.get_expression(),
        "engine_running_event": engine_running_event.get_expression(),
        "engine_rpm_hist_batched": rpm_histogram.get_expression(),
        "batched_speed_rpm_stats": speed_rpm_stats.get_expression(),
    }
    return report, tracked_expressions


def instrument_expression_selectors(tracked_expressions, selector_calls: Counter) -> None:
    for label, expr in tracked_expressions.items():
        original_get_selectors = expr.get_selectors

        def _wrapped(_expr, _label=label, _original=original_get_selectors):
            _ = _expr
            selector_calls[_label] += 1
            return _original()

        expr.get_selectors = types.MethodType(_wrapped, expr)


def install_instrumentation():
    instrumentation = {
        "create_sink_calls": 0,
        "cleanup_calls": 0,
        "solve_calls": [],
        "build_batches": [],
        "basic_event_calls": [],
        "histogram_calls": [],
        "stats_calls": [],
    }

    originals = {
        "create_sink": Report.create_sink,
        "cleanup_temp_tables": Report._cleanup_temp_tables,
        "solve_expressions_batched": Report._solve_expressions_batched,
        "build_batches": report_module.build_batches,
        "basic_event_determine_events": BasicEvent.determine_events.__func__,
        "histogram_determine_aggregations": Histogram.determine_aggregations.__func__,
        "stats_determine_aggregations": StatsAggregator.determine_aggregations.__func__,
    }

    def create_sink_wrapper(config):
        instrumentation["create_sink_calls"] += 1
        return originals["create_sink"](config)

    def cleanup_temp_tables_wrapper(self):
        instrumentation["cleanup_calls"] += 1
        return originals["cleanup_temp_tables"](self)

    def build_batches_wrapper(expressions, batch_size):
        batches = originals["build_batches"](expressions, batch_size)
        instrumentation["build_batches"].append(
            {
                "batch_size": batch_size,
                "expression_count": len(expressions),
                "batch_count": len(batches),
                "batch_aliases": [
                    [getattr(expr, "_alias", expr.__class__.__name__) for expr in batch]
                    for batch in batches
                ],
            }
        )
        return batches

    def solve_expressions_batched_wrapper(self, expressions, pre_filtered_containers_df=None):
        call_info = {
            "expression_count": len(expressions),
            "aliases": [getattr(expr, "_alias", expr.__class__.__name__) for expr in expressions],
            "has_pre_filtered_containers": pre_filtered_containers_df is not None,
        }
        result = originals["solve_expressions_batched"](
            self,
            expressions,
            pre_filtered_containers_df=pre_filtered_containers_df,
        )
        call_info["returned_none"] = result is None
        instrumentation["solve_calls"].append(call_info)
        return result

    def basic_event_wrapper(
        cls,
        spark_session,
        events,
        *,
        solved_df=None,
        query=None,
        solver=None,
        pre_filtered_containers_df=None,
    ):
        instrumentation["basic_event_calls"].append(
            {
                "event_names": [event.get_name() for event in events],
                "has_solved_df": solved_df is not None,
            }
        )
        return originals["basic_event_determine_events"](
            cls,
            spark_session,
            events,
            solved_df=solved_df,
            query=query,
            solver=solver,
            pre_filtered_containers_df=pre_filtered_containers_df,
        )

    def histogram_wrapper(
        cls,
        spark_session,
        aggregations,
        *,
        solved_df=None,
        query=None,
        solver=None,
        pre_filtered_containers_df=None,
    ):
        instrumentation["histogram_calls"].append(
            {
                "aggregation_names": [aggregation.get_name() for aggregation in aggregations],
                "has_solved_df": solved_df is not None,
            }
        )
        return originals["histogram_determine_aggregations"](
            cls,
            spark_session,
            aggregations,
            solved_df=solved_df,
            query=query,
            solver=solver,
            pre_filtered_containers_df=pre_filtered_containers_df,
        )

    def stats_wrapper(
        cls,
        spark_session,
        aggregations,
        *,
        solved_df=None,
        query=None,
        solver=None,
        pre_filtered_containers_df=None,
    ):
        instrumentation["stats_calls"].append(
            {
                "aggregation_names": [aggregation.get_name() for aggregation in aggregations],
                "has_solved_df": solved_df is not None,
            }
        )
        return originals["stats_determine_aggregations"](
            cls,
            spark_session,
            aggregations,
            solved_df=solved_df,
            query=query,
            solver=solver,
            pre_filtered_containers_df=pre_filtered_containers_df,
        )

    Report.create_sink = staticmethod(create_sink_wrapper)
    Report._cleanup_temp_tables = cleanup_temp_tables_wrapper
    Report._solve_expressions_batched = solve_expressions_batched_wrapper
    report_module.build_batches = build_batches_wrapper
    BasicEvent.determine_events = classmethod(basic_event_wrapper)
    Histogram.determine_aggregations = classmethod(histogram_wrapper)
    StatsAggregator.determine_aggregations = classmethod(stats_wrapper)

    return originals, instrumentation


def restore_instrumentation(originals) -> None:
    Report.create_sink = staticmethod(originals["create_sink"])
    Report._cleanup_temp_tables = originals["cleanup_temp_tables"]
    Report._solve_expressions_batched = originals["solve_expressions_batched"]
    report_module.build_batches = originals["build_batches"]
    BasicEvent.determine_events = classmethod(originals["basic_event_determine_events"])
    Histogram.determine_aggregations = classmethod(
        originals["histogram_determine_aggregations"]
    )
    StatsAggregator.determine_aggregations = classmethod(
        originals["stats_determine_aggregations"]
    )

In [0]:
setup_summary = setup_uc_tables()
selector_calls = Counter()
originals, instrumentation = install_instrumentation()

try:
    report, tracked_expressions = build_batched_report()
    instrument_expression_selectors(tracked_expressions, selector_calls)

    stale_temp_table = f"{GOLD_NAMESPACE}.__mda_temp_stale_batch"
    spark.sql(f"DROP TABLE IF EXISTS {stale_temp_table}")
    spark.sql(
        f"CREATE TABLE {stale_temp_table} USING DELTA AS SELECT 1 AS stale_id"
    )

    report.determine_report()

    temp_tables = {
        row.tableName
        for row in spark.sql(f"SHOW TABLES IN {GOLD_NAMESPACE} LIKE '__mda_temp_*'").collect()
    }

    summary = {
        "setup": setup_summary,
        "create_sink_calls": instrumentation["create_sink_calls"],
        "cleanup_calls": instrumentation["cleanup_calls"],
        "build_batches": instrumentation["build_batches"],
        "solve_calls": instrumentation["solve_calls"],
        "selector_calls": dict(selector_calls),
        "event_rows": report.event_dfs["BASIC_EVENT"]["changed"].count(),
        "hist_rows_gt_zero": report.aggregation_dfs["HISTOGRAM"]["changed"].filter(F.col("hist_value") > 0).count(),
        "stats_rows": report.aggregation_dfs["STATS_AGGREGATOR"]["changed"].count(),
        "temp_tables": sorted(temp_tables),
    }

    assert instrumentation["create_sink_calls"] == 1
    assert report._has_sink is True

    assert instrumentation["cleanup_calls"] == 1
    assert "__mda_temp_stale_batch" not in temp_tables
    assert temp_tables

    assert instrumentation["build_batches"]
    assert instrumentation["build_batches"][0]["batch_size"] == 1
    assert instrumentation["build_batches"][0]["batch_count"] >= 2

    assert len(instrumentation["solve_calls"]) == 2
    assert instrumentation["solve_calls"][0]["expression_count"] == len(tracked_expressions)
    assert instrumentation["solve_calls"][0]["returned_none"] is False
    assert instrumentation["solve_calls"][1]["expression_count"] == 0
    assert instrumentation["solve_calls"][1]["returned_none"] is True

    assert all(selector_calls[label] > 0 for label in tracked_expressions)

    assert len(instrumentation["basic_event_calls"]) == 1
    assert instrumentation["basic_event_calls"][0]["has_solved_df"] is True
    assert len(instrumentation["histogram_calls"]) == 1
    assert instrumentation["histogram_calls"][0]["has_solved_df"] is True
    assert len(instrumentation["stats_calls"]) == 1
    assert instrumentation["stats_calls"][0]["has_solved_df"] is True

    assert report.event_dfs["BASIC_EVENT"]["changed"].count() > 0
    assert report.event_metadata_dfs["BASIC_EVENT"].count() == 2
    assert report.aggregation_dfs["HISTOGRAM"]["changed"].filter(F.col("hist_value") > 0).count() > 0
    assert report.aggregation_dfs["STATS_AGGREGATOR"]["changed"].count() > 0
    assert report.aggregation_metadata_dfs["HISTOGRAM"].count() == 1
    assert report.aggregation_metadata_dfs["STATS_AGGREGATOR"].count() == 1

    # container_dimension_df: verify STOP_TS is exposed as stop_ts (mapped from source end_ts)
    assert report.container_dimension_df is not None, "container_dimension_df should not be None"
    container_dim_cols = report.container_dimension_df.columns
    assert "stop_ts" in container_dim_cols, (
        f"Expected 'stop_ts' in container_dimension_df columns (mapped from end_ts), got: {container_dim_cols}"
    )
    assert "uut_id" in container_dim_cols
    assert "start_ts" in container_dim_cols
    assert "container_id" in container_dim_cols
    assert report.container_dimension_df.count() > 0

    print("Batched pipeline validation passed")
    print(summary)
finally:
    restore_instrumentation(originals)

In [0]:
report.event_dfs["BASIC_EVENT"]["changed"].orderBy("container_id", "event_name").show(20, truncate=False)
report.aggregation_dfs["HISTOGRAM"]["changed"].filter(F.col("hist_value") > 0).show(20, truncate=False)
report.aggregation_dfs["STATS_AGGREGATOR"]["changed"].show(20, truncate=False)